# A1 · Foundations

**IMSE 774 · Neural Networks · Formative Assignment 1 · Weeks 1–2**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harunpirim/IMSE774-NeuralNetworks/blob/main/assignments/A1-foundations.ipynb)

This assignment spans the first two weeks of the course:

- **Week 1 — [Introduction and Supervised Learning](https://harunpirim.github.io/IMSE774-NeuralNetworks/notes/01-introduction.html)**: Prince, *Understanding Deep Learning*, Chapters 1–2.
- **Week 2 — [Mathematical and Computational Background](https://harunpirim.github.io/IMSE774-NeuralNetworks/notes/02-math-background.html)**: Prince, Appendix B (§B.3–§B.5) and Chapter 2 revisited.

### What "formative" means here

The purpose of this assignment is feedback, not ranking. It exists to show you, and me, what has landed and what has not *before* Week 3 builds on it. Almost every task ends in a **self-check** that tells you immediately whether the idea is right; the written prompts ask for the reasoning behind it. An honest wrong answer with your reasoning is more useful to both of us than a right answer you cannot explain.

### How to work

1. **File → Save a copy in Drive**, so your edits persist.
2. Run the **Setup** cell first. Every later cell depends on the names it defines.
3. Work top to bottom. Blanks look like `None`, `...`, or `# YOUR CODE HERE`. Written prompts have a cell that starts with *Your answer:* — write directly in it (double-click to edit).
4. Run the check cell under each task. ✅ means the idea landed; ❌ comes with a hint that says where to look; ⬜ means you have not answered yet.
5. When you are done, **Runtime → Run all**, confirm there are no errors, and run the **Summary** at the bottom.
6. **Submit on Blackboard under Assignment A1.** Download your notebook with **File → Download → Download .ipynb** and upload that file, named `A1_<lastname>.ipynb`. Do not submit a link to your Drive copy. The deadline is posted on Blackboard.

Budget about **3–4 hours**. Times next to each task are rough. You may use any resource, including the course notebooks and the textbook, but write your own code and your own sentences.

| Part | Covers | Tasks |
|:--|:--|:--|
| **A** | What learning is: paradigms, inputs, latent variables, reinforcement learning, ethics | A1–A6 |
| **B** | The supervised recipe: model, loss, training three ways, testing | B1–B4 |
| **C** | Appendix B in code: shapes, norms, products, subspaces, eigenspectrum, one layer, matrix calculus, conditioning | C1–C8 |
| **D** | Self-assessment | D1 |

## Setup

Run this first. It imports the libraries, fixes the random seeds, and defines the self-check helpers used throughout.

In [ ]:
import hashlib
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(774)
np.random.seed(774)
rng = np.random.default_rng(774)

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.frameon": False,
})

NDSU_GREEN = "#005643"
NDSU_GOLD  = "#FFC72C"
ACCENT     = "#C1121F"

# ---------------------------------------------------------------------------
# Self-check helpers. A failed check is feedback, never a crash.
# ---------------------------------------------------------------------------
_RESULTS = {}

def _key(s):
    return " ".join(str(s).strip().lower().replace("_", " ").split())

def _fingerprint(s):
    return hashlib.sha256(_key(s).encode()).hexdigest()[:12]

def check(label, ok, hint=""):
    """Record and print one pass/fail line. Pass ok=None for 'not answered yet'."""
    if ok is None:
        _RESULTS[label] = False
        print("⬜ " + label + "   ← not answered yet")
        return False
    ok = bool(ok)
    _RESULTS[label] = ok
    print(("✅ " if ok else "❌ ") + label + ("" if ok or not hint else "   ← " + hint))
    return ok

def _join(*parts):
    """'a|b|c' for the fingerprint checks, or None if any part is still blank."""
    return None if any(p is None for p in parts) else "|".join(str(p) for p in parts)

def check_answer(label, given, fingerprint, hint=""):
    """Compare a short written answer with a fingerprint of the expected one.
    Case, spacing, and underscores are ignored."""
    if given is None or (isinstance(given, str) and not given.strip()):
        _RESULTS[label] = False
        print("⬜ " + label + "   ← not answered yet")
        return False
    return check(label, _fingerprint(given) == fingerprint, hint)

def check_close(label, got, want, tol=1e-3, hint=""):
    """Numerical comparison with a relative and absolute tolerance."""
    if got is None:
        _RESULTS[label] = False
        print("⬜ " + label + "   ← not answered yet")
        return False
    try:
        ok = np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float),
                         rtol=tol, atol=tol)
    except Exception:
        ok = False
    return check(label, ok, hint)

def summary():
    parts = {}
    for label, ok in _RESULTS.items():
        part = label[0]
        parts.setdefault(part, [0, 0])
        parts[part][0] += int(ok)
        parts[part][1] += 1
    print(f"{'part':<6}{'passed':>8}{'of':>6}")
    print("-" * 20)
    for part in sorted(parts):
        p, n = parts[part]
        print(f"{part:<6}{p:>8}{n:>6}")
    p = sum(v[0] for v in parts.values()); n = sum(v[1] for v in parts.values())
    print("-" * 20)
    print(f"{'all':<6}{p:>8}{n:>6}")
    missing = [k for k, v in _RESULTS.items() if not v]
    if missing:
        print("\nNot yet passing:", ", ".join(missing))

print("Setup complete. torch", torch.__version__, "· numpy", np.__version__)

## Part A — What learning is

*Week 1 note; Prince, Chapter 1.*

### A1 · Four words that are not synonyms *(5 min)*

Press coverage uses *artificial intelligence*, *machine learning*, and *deep learning* interchangeably. Prince does not, and figure 1.1 of the book draws the containment.

Order the three fields from **broadest to narrowest**, and say what kind of thing *deep learning* is in Prince's usage: a **model**, or a **procedure** applied to a model.

In [ ]:
# Use exactly these strings: "artificial intelligence", "machine learning", "deep learning"
broad_to_narrow = [None, None, None]

# One of: "a model", "a procedure"
deep_learning_is = None

In [ ]:
check_answer("A1 containment", _join(*broad_to_narrow),
             "048c70b5103f",
             "Figure 1.1: which field contains which? Logic and search are AI but not ML.")
check_answer("A1 what deep learning is", deep_learning_is,
             "1fd3e68a87f7",
             "Re-read the four definitions on p. 1. The deep neural network is the model; what is deep learning?")

### A2 · Classify the problem before you pick the model *(15 min)*

For each task below, name the **learning paradigm** and, if it is supervised, the **output type** from the taxonomy of Prince figure 1.2. Use the strings in the two option lists exactly; write `"n/a"` for the output type when the task is not supervised.

| Key | Task |
|:--|:--|
| `a` | Predict the remaining useful life, in hours, of a bearing from its vibration spectrum |
| `b` | Group warehouse SKUs by the shape of their weekly demand pattern, with no labels |
| `c` | Decide which job to dispatch next on a shop floor, given a reward observed at the end of each shift for low makespan |
| `d` | Flag a weld as pass or fail from an X-ray image |
| `e` | Predict a new polymer's melting point *and* glass-transition temperature from its molecular structure |
| `f` | Generate synthetic defect images to augment a small training set |
| `g` | Identify which of eight failure modes a pump is in from its sensor log |

In [ ]:
paradigm_options = ["supervised", "unsupervised", "reinforcement"]
output_options   = ["regression", "multivariate regression",
                    "binary classification", "multiclass classification", "n/a"]

answers_A2 = {
    "a": (None, None),   # (paradigm, output type)
    "b": (None, None),
    "c": (None, None),
    "d": (None, None),
    "e": (None, None),
    "f": (None, None),
    "g": (None, None),
}

In [ ]:
for k, (p, o) in answers_A2.items():
    if p is not None and p not in paradigm_options:
        print(f"A2({k}): {p!r} is not one of {paradigm_options}")
    if o is not None and o not in output_options:
        print(f"A2({k}): {o!r} is not one of {output_options}")

check_answer("A2(a)", _join(*answers_A2["a"]), "6f22e62c4a91", "Hours are a continuous quantity; is there a label per training example?")
check_answer("A2(b)", _join(*answers_A2["b"]), "263c2548e7b3", "No labels, and the goal is to describe structure in the inputs.")
check_answer("A2(c)", _join(*answers_A2["c"]), "c979e307aee4", "Actions, a delayed reward, and no labeled correct action.")
check_answer("A2(d)", _join(*answers_A2["d"]), "ebfeca79a75a", "Two categories, with a labeled outcome per image.")
check_answer("A2(e)", _join(*answers_A2["e"]), "a74d50b4b0ee", "Two continuous outputs per input, like figure 1.2b.")
check_answer("A2(f)", _join(*answers_A2["f"]), "263c2548e7b3", "Synthesizing new examples that resemble the data is generative modeling, §1.2.1.")
check_answer("A2(g)", _join(*answers_A2["g"]), "7a3f714867e2", "Probabilities over N > 2 categories.")

**Written.** Task `f` is the odd one out: it is *unsupervised* work done in service of a *supervised* problem. In two or three sentences, explain what a latent-variable generative model buys you when the labeled dataset is small. Use the terms *latent variables* and *labels* (Prince §1.2.2–§1.2.3).

*Your answer:*

### A3 · Inputs are the hard part *(15 min)*

Prince §1.1.2 argues that what makes a problem hard is the **input**: its size, whether it has a fixed length, and its internal structure. Put numbers on that.

For each input below, fill in the **number of values in the input vector**. Then the cell computes how many parameters a single fully connected layer with 1,000 hidden units would need to consume it, which is `(d + 1) × 1000` (one weight per input plus a bias per unit).

In [ ]:
HIDDEN = 1_000

input_size = {
    "8 sensor readings per part (tabular)":         None,
    "640 x 480 grayscale inspection image":         None,
    "10 s vibration record sampled at 25.6 kHz":    None,
    "5 s of 1080p RGB video at 30 frames per s":    None,   # 1080p is 1920 x 1080 pixels
}

for name, d in input_size.items():
    params = None if d is None else (d + 1) * HIDDEN
    print(f"{name:<46}{'—' if d is None else f'{d:>14,}'}{'' if params is None else f'{params:>20,} parameters'}")

Now name the **internal structure** each input has, which a fully connected layer throws away. Use one of `"none"`, `"1D sequence"`, `"2D grid"`, `"3D grid"`.

In [ ]:
structure = {
    "tabular":   None,
    "image":     None,
    "vibration": None,
    "video":     None,
}

In [ ]:
_want = [8, 640 * 480, 10 * 25_600, 5 * 30 * 1920 * 1080 * 3]
for (name, d), w in zip(input_size.items(), _want):
    check_close("A3 size: " + name.split(" (")[0][:28], d, w, tol=0,
                hint="Count every stored number: samples × channels × pixels.")
check_answer("A3 structure", _join(*[structure[k] for k in ["tabular", "image", "vibration", "video"]]),
             "b8fa9e3c14fe",
             "Which inputs have neighbors that are related? Along how many axes?")

**Written.** The video row is why "flatten it and use a fully connected layer" stops being an option almost immediately. Which architecture from later in the course exploits the 2D grid structure of an image, and in one sentence, *what* about that structure does it exploit? (Prince §1.1.2; Chapter 10.)

*Your answer:*

### A4 · Latent variables: the data are smaller than they look *(20 min)*

Prince §1.2.2 makes an argument you can test. A $32\times32$ RGB image is a point in a space with $256^{3072}$ elements, yet the photographs a camera can take occupy a vanishingly small, highly structured region of it. Generative models are in the business of finding that region.

Run the first cell to see eight uniformly random images. None of them will ever be a photograph.

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(11, 1.6))
for ax in axes:
    ax.imshow(rng.integers(0, 256, size=(32, 32, 3), dtype=np.uint8))
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.tight_layout(); plt.show()

n_values = 32 * 32 * 3
print(f"points in image space: 256^{n_values}, about 10^{n_values * np.log10(256):.0f}")

Now a dataset with a *known* latent structure. Imagine an inspection rig photographing a single bright feature. Every image is generated by exactly **three numbers**: the feature's position $(c_x, c_y)$ and its width $\sigma$. The images are $16\times16$, so each one has 256 values.

Your task: measure how many dimensions the data *actually* occupy. Center the data matrix, compute its **singular values** (Week 2, §B.3.7: the singular values are the semi-axes of the ellipsoid the data fill), and find the smallest number of components $k$ whose squared singular values account for **95% of the total**.

In [ ]:
def blob_image(cx, cy, sigma, size=16):
    r = np.arange(size)
    gx, gy = np.meshgrid(r, r)
    return np.exp(-((gx - cx) ** 2 + (gy - cy) ** 2) / (2 * sigma ** 2))

n = 1000
latents = np.column_stack([rng.uniform(3, 12, n), rng.uniform(3, 12, n), rng.uniform(1.0, 2.5, n)])
images = np.stack([blob_image(*l).ravel() for l in latents])      # n x 256, one image per row
print("data matrix:", images.shape)

fig, axes = plt.subplots(1, 8, figsize=(11, 1.6))
for ax, img in zip(axes, images[:8]):
    ax.imshow(img.reshape(16, 16), cmap="gray"); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.tight_layout(); plt.show()

# --- YOUR CODE HERE --------------------------------------------------------
centered = None                     # subtract the mean image from every row
s = None                            # singular values of `centered`  (np.linalg.svd(..., compute_uv=False))
energy = None                       # cumulative fraction of sum(s**2), a vector of length 256
k95 = None                          # smallest k with energy[k-1] >= 0.95
# ---------------------------------------------------------------------------

if s is not None:
    plt.figure(figsize=(6, 3.2))
    plt.semilogy(np.arange(1, len(s) + 1), s, ".", color=NDSU_GREEN)
    plt.axvline(k95, color=ACCENT, ls="--", label=f"95% of energy at k = {k95}")
    plt.xlabel("component"); plt.ylabel("singular value"); plt.legend(); plt.show()
    print(f"components for 95% of the energy: {k95} of {images.shape[1]}, generated from {latents.shape[1]} latent numbers")

In [ ]:
_c = images - images.mean(axis=0)
_s = np.linalg.svd(_c, compute_uv=False)
_e = np.cumsum(_s ** 2) / np.sum(_s ** 2)
check_close("A4 singular values", None if s is None else s[:5], _s[:5], tol=1e-4,
            hint="Center first (subtract the column means), then np.linalg.svd(centered, compute_uv=False).")
check_close("A4 k95", k95, int(np.argmax(_e >= 0.95)) + 1, tol=0,
            hint="energy = np.cumsum(s**2) / np.sum(s**2); k95 is the first index (1-based) where it reaches 0.95.")

**Written.** Three numbers generated every image, yet you needed more than three components to capture 95% of the energy. Why is $k_{95}$ larger than 3, and why is it still far smaller than 256? What does this say about why the mapping from latent variables to data is a *deep network* rather than a matrix? (Prince §1.2.2, figure 1.10.)

*Your answer:*

### A5 · Reinforcement learning in its smallest form *(25 min)*

Prince §1.3 names two problems that only reinforcement learning has: **temporal credit assignment** and the **exploration–exploitation trade-off**. The smallest experiment that shows the second is a $k$-armed bandit: $k$ levers with unknown mean payoffs, and a policy that pulls the best-known lever with probability $1-\epsilon$ and a random lever otherwise.

The simulation below is written except for its two essential lines. Fill them in:

1. **Action selection.** For every run, choose a random arm when `explore` is `True`, otherwise the arm with the highest current estimate `Q`.
2. **The incremental mean.** After observing reward `r` from arm `a`, update the estimate: $Q_a \leftarrow Q_a + \frac{1}{N_a}(r - Q_a)$.

In [ ]:
def run_bandit(eps, n_runs=200, k=10, steps=3000, seed=0):
    rng_b = np.random.default_rng(seed)
    q_true = rng_b.normal(0, 1, size=(n_runs, k))   # true mean payoff of every arm, per run
    best = q_true.argmax(axis=1)                     # the arm we hope the agent finds
    Q = np.zeros((n_runs, k))                        # running estimate of each arm's payoff
    N = np.zeros((n_runs, k))                        # how often each arm has been pulled
    runs = np.arange(n_runs)
    avg_reward, pct_optimal = np.zeros(steps), np.zeros(steps)

    for t in range(steps):
        explore = rng_b.random(n_runs) < eps                    # shape (n_runs,), True = explore
        random_arm = rng_b.integers(0, k, n_runs)

        a = None                                                 # YOUR CODE HERE (1): shape (n_runs,)
        if a is None:
            raise NotImplementedError("fill in the action selection, YOUR CODE HERE (1)")

        r = rng_b.normal(q_true[runs, a], 1.0)                   # noisy reward from the chosen arms
        N[runs, a] += 1

        # YOUR CODE HERE (2): update Q[runs, a] with the incremental mean

        avg_reward[t] = r.mean()
        pct_optimal[t] = (a == best).mean()
    return avg_reward, pct_optimal

results = {}
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for eps, color in [(0.0, ACCENT), (0.01, NDSU_GREEN), (0.1, "#1864AB"), (0.5, "#7C3AED")]:
    try:
        reward, optimal = run_bandit(eps)
    except Exception as e:
        print(f"eps = {eps}: not running yet ({type(e).__name__}: {e})"); break
    results[eps] = (reward, optimal)
    label = "greedy ($\\epsilon = 0$)" if eps == 0 else f"$\\epsilon = {eps}$"
    axes[0].plot(reward, color=color, lw=1.1, label=label)
    axes[1].plot(100 * optimal, color=color, lw=1.1, label=label)
axes[0].set_xlabel("step"); axes[0].set_ylabel("average reward"); axes[0].legend(fontsize=8)
axes[1].set_xlabel("step"); axes[1].set_ylabel("% optimal action"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

for eps, (reward, optimal) in results.items():
    print(f"eps = {eps:<5} optimal action at step 100: {100*optimal[99]:5.1f}%   at step 1000: {100*optimal[999]:5.1f}%   at the end: {100*optimal[-1]:5.1f}%")

Answer from the printed numbers, then from the theory.

In [ ]:
# Which epsilon chooses the optimal arm most often at step 100, at step 1000, and at the end? (floats: 0.0, 0.01, 0.1, 0.5)
leader_at_100  = None
leader_at_1000 = None
leader_at_end  = None

# Which of Prince's two RL-specific problems does the bandit NOT have?
# One of: "temporal credit assignment", "exploration-exploitation trade-off"
problem_the_bandit_lacks = None

In [ ]:
_ok = len(results) == 4
check("A5 simulation runs for all four epsilons", _ok, "Both YOUR CODE HERE lines must be filled before the loop completes.")
if _ok:
    _last = {e: v[1][-500:].mean() for e, v in results.items()}
    check("A5 exploration beats pure greed", _last[0.0] < _last[0.1],
          "A greedy agent locks onto the first arm that pays; check your np.where and argmax axis.")
    check("A5 too much exploration also loses", _last[0.5] < _last[0.1],
          "With eps = 0.5, half of all pulls are random regardless of what has been learned.")
    def _leader(t):
        return max(results, key=lambda e: results[e][1][t - 1])
    check_close("A5 leader at step 100",  leader_at_100,  _leader(100),  tol=0, hint="Read the printed table.")
    check_close("A5 leader at step 1000", leader_at_1000, _leader(1000), tol=0, hint="Read the printed table.")
    check_close("A5 leader at the end",   leader_at_end,  _leader(3000), tol=0, hint="Read the printed table.")
check_answer("A5 problem the bandit lacks", problem_the_bandit_lacks,
             "659445a652eb",
             "In a bandit the reward arrives immediately after the pull. Which problem needs a delay?")

**Written.** Explain the crossover the theory predicts: over a long enough horizon $\epsilon = 0.01$ overtakes $\epsilon = 0.1$, while $\epsilon = 0.5$ never competes. Use the words *exploration tax* and *identify the best arm*. Then say, in one sentence, why teaching a robot to walk is harder than this bandit.

*Your answer:*

### A6 · Ethics, concretely *(20 min)*

Prince §1.4 raises five concerns and asks instructors to raise them with students. Pick **one** concern and **one** deep learning application in your own field. Write one paragraph that

1. names a concrete way the concern could materialize in that application,
2. proposes one design or process decision that would reduce it, and
3. is specific enough that a classmate could disagree with you.

Set `concern` to your choice so the check can confirm it is one of the five. The paragraph itself is read by a person, not a script.

In [ ]:
five_concerns = ["bias and fairness", "explainability", "weaponizing AI",
                 "concentrating power", "existential risk"]

concern = None          # one of five_concerns
application = None      # a short phrase, e.g. "visual weld inspection on an automotive line"

*Your paragraph:*

In [ ]:
check("A6 concern is one of the five", None if concern is None else concern in five_concerns,
      "Use the exact strings in five_concerns.")
check("A6 application named", None if application is None else isinstance(application, str) and len(application.strip()) > 5,
      "Name a specific application in your field.")

## Part B — The supervised recipe

*Week 1 note (Session 2) and Week 2 note (§7–§8); Prince, Chapter 2.*

Everything in supervised learning appears in 1D linear regression, which is why the notes walk it through twice. Here you build it once yourself, on the **cure-temperature dataset** from the Week 2 note: the cure temperature of a composite panel against the tensile strength of the finished part.

### B1 · The vocabulary, precisely *(10 min)*

Match each statement to the term it defines. Terms: `model`, `parameters`, `loss`, `training`, `inference`, `test data`, `underfitting`, `overfitting`.

In [ ]:
vocab = {
    1: None,  # An equation of fixed form mapping input to output; it represents a whole family of relationships
    2: None,  # The numbers that pick one member of that family
    3: None,  # A scalar that quantifies the mismatch between predictions and the observed outputs on the training set
    4: None,  # Searching the family for the member that minimizes that scalar
    5: None,  # Computing the output for a new input
    6: None,  # Examples the model never saw, used to see whether it generalizes
    7: None,  # The model is too simple to represent the true relationship
    8: None,  # The model describes statistical peculiarities of the training set that are atypical
}

In [ ]:
check_answer("B1 statement 1", vocab[1], "9372c470eead", "Prince §2.1–§2.2.4.")
check_answer("B1 statement 2", vocab[2], "f528212a496d", "Prince §2.1–§2.2.4.")
check_answer("B1 statement 3", vocab[3], "2ea71c18131a", "Prince §2.1–§2.2.4.")
check_answer("B1 statement 4", vocab[4], "c2fb788c7dee", "Prince §2.1–§2.2.4.")
check_answer("B1 statement 5", vocab[5], "925733dafd74", "Prince §2.1–§2.2.4.")
check_answer("B1 statement 6", vocab[6], "916f0027a575", "Prince §2.1–§2.2.4.")
check_answer("B1 statement 7", vocab[7], "5a43f1e23e0f", "Prince §2.1–§2.2.4.")
check_answer("B1 statement 8", vocab[8], "a1f32f577f69", "Prince §2.1–§2.2.4.")

### B2 · The loss and its gradient *(30 min)*

The model is a line, $y = f[x, \boldsymbol{\phi}] = \phi_0 + \phi_1 x$, and the loss is least squares:

$$
L[\boldsymbol{\phi}] = \sum_{i=1}^{I} (\phi_0 + \phi_1 x_i - y_i)^2 .
$$

Stack the inputs into a design matrix $\mathbf{X} \in \mathbb{R}^{I\times 2}$ whose first column is ones and whose second column is $x$. Then $L[\boldsymbol{\phi}] = \|\mathbf{X}\boldsymbol{\phi} - \mathbf{y}\|^2$ and (Week 2 note, §8.3)

$$
\frac{\partial L}{\partial \boldsymbol{\phi}} = 2\,\mathbf{X}^T(\mathbf{X}\boldsymbol{\phi} - \mathbf{y}).
$$

**Part 1 (UDL problem 2.1, by hand).** Differentiate the scalar form of the loss with respect to $\phi_0$ and $\phi_1$. Write both expressions in the markdown cell, then implement them in code as `dL_dphi0` and `dL_dphi1`.

**Part 2.** Implement the matrix forms `loss(phi, X, y)` and `grad(phi, X, y)`. The checks compare your gradient with your scalar derivatives, with finite differences, and with PyTorch autograd.

In [ ]:
I = 24
temp = np.random.uniform(140, 200, I)                          # cure temperature, deg C
strength = 12.0 + 0.42 * temp + np.random.normal(0, 2.5, I)    # tensile strength, MPa

X = np.column_stack([np.ones(I), temp])                        # design matrix, I x 2
y = strength

plt.figure(figsize=(5.5, 3.4))
plt.scatter(temp, strength, s=30, color=ACCENT)
plt.xlabel("cure temperature (°C)"); plt.ylabel("tensile strength (MPa)"); plt.show()
print("X:", X.shape, " y:", y.shape)

*Your derivation* (edit this cell; `$...$` renders LaTeX):

$\dfrac{\partial L}{\partial \phi_0} = \; ?$

$\dfrac{\partial L}{\partial \phi_1} = \; ?$

In [ ]:
def loss(phi, X, y):
    """Least-squares loss ||X phi - y||^2."""
    return None                                  # YOUR CODE HERE

def grad(phi, X, y):
    """Gradient of the loss with respect to phi, shape (2,)."""
    return None                                  # YOUR CODE HERE

def scalar_grad(phi, x, y):
    """Your hand-derived expressions, using only x (the temperatures), y, phi[0], phi[1]."""
    dL_dphi0 = None                              # YOUR CODE HERE
    dL_dphi1 = None                              # YOUR CODE HERE
    return None if dL_dphi0 is None else np.array([dL_dphi0, dL_dphi1])

phi_test = np.array([10.0, 0.3])
print("loss at phi_test       :", loss(phi_test, X, y))
print("matrix gradient        :", grad(phi_test, X, y))
print("hand-derived gradient  :", scalar_grad(phi_test, temp, y))

In [ ]:
_L = lambda p: np.sum((X @ p - y) ** 2)
check_close("B2 loss", loss(phi_test, X, y), _L(phi_test), tol=1e-6,
            hint="Sum of squared residuals: np.sum((X @ phi - y) ** 2).")

_g = grad(phi_test, X, y)
_h = 1e-4
_fd = np.array([(_L(phi_test + _h * e) - _L(phi_test - _h * e)) / (2 * _h) for e in np.eye(2)])
check_close("B2 gradient vs finite differences", _g, _fd, tol=1e-4,
            hint="2 * X.T @ (X @ phi - y); mind the order of the product.")

_p = torch.tensor(phi_test, requires_grad=True)
_Xt, _yt = torch.tensor(X), torch.tensor(y)
torch.sum((_Xt @ _p - _yt) ** 2).backward()
check_close("B2 gradient vs autograd", _g, _p.grad.numpy(), tol=1e-6,
            hint="Autograd disagrees with your matrix gradient.")

check_close("B2 hand derivation matches matrix form", scalar_grad(phi_test, temp, y), _g, tol=1e-6,
            hint="Both derivatives carry a factor 2; the second is weighted by x_i.")

### B3 · Training, three ways *(30 min)*

Linear regression has a closed form, but every model after Chapter 3 does not, so the notes fit it three ways and compare. Do the same.

1. **Closed form (UDL problem 2.2).** Set your gradient to zero: $\mathbf{X}^T\mathbf{X}\,\hat{\boldsymbol{\phi}} = \mathbf{X}^T\mathbf{y}$, the *normal equations*. **Solve** them with `np.linalg.solve`; never invert.
2. **Gradient descent.** Walk downhill with your `grad` from B2. The Week 2 note shows that on the raw temperatures this stalls, so first **standardize** the input, $z = (x - \mu)/\sigma$, and fit $\boldsymbol{\psi}$ on the design matrix $\mathbf{Z} = [\mathbf{1}, z]$. Use the learning rate given, which is 90% of the largest stable step.
3. **PyTorch.** One `nn.Linear(1, 1)` layer trained by SGD on the standardized input. Its `weight` is $\psi_1$ and its `bias` is $\psi_0$.

The three answers live on different scales. Convert the standardized fits back to the raw scale with
$\phi_1 = \psi_1 / \sigma$ and $\phi_0 = \psi_0 - \psi_1 \mu / \sigma$, so that all three can be compared.

In [ ]:
# 1. Closed form -------------------------------------------------------------
phi_closed = None                                       # YOUR CODE HERE  (np.linalg.solve)

# 2. Gradient descent on the standardized input ------------------------------
mu, sigma = temp.mean(), temp.std()
z = (temp - mu) / sigma
Z = np.column_stack([np.ones(I), z])
lr = 0.9 * 2 / np.linalg.eigvalsh(2 * Z.T @ Z).max()    # 90% of the largest stable step size

psi = np.zeros(2)
history = []
for step in range(200):
    pass                                                # YOUR CODE HERE: one gradient step on psi, then history.append(loss(psi, Z, y))

def unstandardize(psi):
    """Map parameters fitted on z back to the raw temperature scale."""
    return None                                         # YOUR CODE HERE: np.array([phi_0, phi_1])

phi_gd = None if psi is None else unstandardize(psi)

# 3. PyTorch -------------------------------------------------------------------
Zt = torch.tensor(z, dtype=torch.float32).unsqueeze(1)
Yt = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

model = nn.Linear(1, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()
for epoch in range(500):
    pass                                                # YOUR CODE HERE: zero_grad, forward, loss, backward, step

psi_torch = np.array([model.bias.item(), model.weight.item()])
phi_torch = unstandardize(psi_torch)

print(f"{'method':<22}{'phi_0':>10}{'phi_1':>10}{'loss':>12}")
for name, p in [("closed form", phi_closed), ("gradient descent", phi_gd), ("PyTorch (SGD)", phi_torch)]:
    if p is None:
        print(f"{name:<22}{'—':>10}")
    else:
        print(f"{name:<22}{p[0]:>10.4f}{p[1]:>10.4f}{loss(p, X, y):>12.4f}")

In [ ]:
_phi_star = np.linalg.lstsq(X, y, rcond=None)[0]
_best = np.sum((X @ _phi_star - y) ** 2)
check_close("B3 closed form", phi_closed, _phi_star, tol=1e-6,
            hint="np.linalg.solve(X.T @ X, X.T @ y).")
check("B3 gradient descent reached the optimum",
      None if phi_gd is None else loss(phi_gd, X, y) <= 1.001 * _best,
      "Each step is psi = psi - lr * grad(psi, Z, y); did unstandardize map psi back correctly?")
check("B3 PyTorch reached the optimum",
      None if phi_torch is None else loss(phi_torch, X, y) <= 1.001 * _best,
      "The loop needs opt.zero_grad(), a forward pass, loss_fn(...).backward(), opt.step().")
check("B3 history is decreasing", None if not history else len(history) == 200 and history[-1] < history[0],
      "Append loss(psi, Z, y) after every step.")

**Written.** Why is the gradient descent answer expected to *agree* with the closed form here, and what property of a shallow or deep network (Chapter 3 onward) removes the closed form? One or two sentences. (Prince §2.2.3; UDL problem 2.2.)

*Your answer:*

### B4 · Testing: training loss cannot choose the model *(25 min)*

Training loss alone tells us nothing. Below is a fresh batch of 200 parts from the same process. Fit polynomials of several degrees to the 24 training panels (on the standardized input, using the **training** mean and standard deviation for both sets), and record the **mean squared error** on the training set and on the test set for each degree.

Degree 0 is a horizontal line, degree 1 is your model from B3, and degree 12 has enough freedom to weave through most of the training points.

In [ ]:
temp_test = np.random.uniform(140, 200, 200)
strength_test = 12.0 + 0.42 * temp_test + np.random.normal(0, 2.5, 200)

z_train = (temp - mu) / sigma
z_test = (temp_test - mu) / sigma            # training statistics: the test set must never leak into the fit

degrees = [0, 1, 2, 3, 6, 12]
train_mse, test_mse = [], []
for degree in degrees:
    coef = np.polyfit(z_train, strength, degree)
    train_mse.append(None)                    # YOUR CODE HERE: mean squared error of np.polyval(coef, z_train) against strength
    test_mse.append(None)                     # YOUR CODE HERE: the same on the test set

if None not in train_mse:
    plt.figure(figsize=(6, 3.4))
    plt.semilogy(degrees, train_mse, "o-", color=NDSU_GREEN, label="train")
    plt.semilogy(degrees, test_mse, "s-", color=ACCENT, label="test")
    plt.xlabel("polynomial degree"); plt.ylabel("mean squared error"); plt.legend(); plt.show()
    for d, tr, te in zip(degrees, train_mse, test_mse):
        print(f"degree {d:>2}   train {tr:8.2f}   test {te:10.2f}")

best_degree = None                            # the degree you would deploy

In [ ]:
_tr, _te = [], []
for _d in degrees:
    _c = np.polyfit(z_train, strength, _d)
    _tr.append(np.mean((np.polyval(_c, z_train) - strength) ** 2))
    _te.append(np.mean((np.polyval(_c, z_test) - strength_test) ** 2))
check_close("B4 training MSE", None if None in train_mse else train_mse, _tr, tol=1e-6,
            hint="np.mean((prediction - target) ** 2).")
check_close("B4 test MSE", None if None in test_mse else test_mse, _te, tol=1e-6,
            hint="Evaluate on z_test against strength_test.")
check("B4 training loss falls with degree", None if None in train_mse else all(np.diff(train_mse) <= 1e-9),
      "A more expressive model always fits the training set at least as well.")
check_close("B4 degree to deploy", best_degree, degrees[int(np.argmin(_te))], tol=0,
            hint="The model we want is at the bottom of the test curve, not the training curve.")

**Written.** Name which degrees are *underfitting* and which are *overfitting*, and explain in two sentences why the training loss cannot be used to choose between them. What would you do if you could not manufacture 200 extra parts? (Prince §2.2.4; Chapter 8.)

*Your answer:*

## Part C — Appendix B in code

*Week 2 note; Prince, Appendix B §B.3–§B.5.*

Every fully connected layer from Chapter 3 onward computes $\mathbf{y} = \boldsymbol{\phi}_0 + \boldsymbol{\Phi}\mathbf{z}$ and applies a nonlinearity. The tasks below make sure that object, and the algebra around it, are comfortable.

### C1 · Shapes, and why the transpose reverses *(20 min)*

*(Week 2 exercises 1 and 2.)* A quality inspection station records **8 sensor readings** for each of **500 parts**, and a layer produces **3 outputs per part**.

Write down, as tuples, the shape of the data matrix $\mathbf{Z}$ with **one part per row**, of the parameter matrix $\boldsymbol{\Phi}$ in Prince's convention $\mathbf{y} = \boldsymbol{\phi}_0 + \boldsymbol{\Phi}\mathbf{z}$ (one column vector $\mathbf{z}$ at a time), of the bias $\boldsymbol{\phi}_0$, and of the output for the whole batch. The check builds the layer in PyTorch and compares.

In [ ]:
shape_Z      = None      # e.g. (rows, cols)
shape_Phi    = None
shape_phi0   = None
shape_output = None

In [ ]:
_layer = nn.Linear(8, 3)
_Zb = torch.randn(500, 8)
def _shape_check(label, given, want, hint):
    check(label, None if given is None else tuple(given) == tuple(want), hint)
_shape_check("C1 shape of Z", shape_Z, _Zb.shape, "One part per row: (parts, readings).")
_shape_check("C1 shape of Phi", shape_Phi, _layer.weight.shape, "Phi maps 8 inputs to 3 outputs; Prince's Phi is (outputs, inputs), exactly nn.Linear's weight.")
_shape_check("C1 shape of phi_0", shape_phi0, _layer.bias.shape, "One bias per output.")
_shape_check("C1 shape of output", shape_output, _layer(_Zb).shape, "One row of 3 outputs per part.")

With parts stored as rows, PyTorch computes the whole batch as $\mathbf{Y} = \mathbf{Z}\boldsymbol{\Phi}^T + \boldsymbol{\phi}_0^T$. That $\boldsymbol{\Phi}^T$ comes from the transpose-of-a-product rule, $(\mathbf{A}\mathbf{B})^T = \mathbf{B}^T\mathbf{A}^T$.

Verify the three-factor version $(\mathbf{A}\mathbf{B}\mathbf{C})^T = \mathbf{C}^T\mathbf{B}^T\mathbf{A}^T$ numerically, and implement the matrix product as the explicit double sum of (B.10), $C_{ij} = \sum_d A_{id} B_{dj}$, to confirm it reproduces `@`.

In [ ]:
A = rng.standard_normal((3, 4)); B = rng.standard_normal((4, 5)); C = rng.standard_normal((5, 2))

lhs = (A @ B @ C).T
rhs = None                                   # YOUR CODE HERE: the product of transposes, in the right order

def matmul_loops(A, B):
    """Equation (B.10) written out as loops. Return a NumPy array."""
    out = np.zeros((A.shape[0], B.shape[1]))
    # YOUR CODE HERE
    return out

print("(ABC)^T shape:", lhs.shape, "  rhs shape:", None if rhs is None else rhs.shape)

In [ ]:
check_close("C1 (ABC)^T = C^T B^T A^T", rhs, lhs, tol=1e-9,
            hint="Apply (AB)^T = B^T A^T twice; only one ordering has compatible shapes.")
check_close("C1 loop product matches @", matmul_loops(A, B), A @ B, tol=1e-9,
            hint="Three nested loops over i, j, and the shared index d.")

**Written.** In one sentence about shapes, why could no other ordering of $\mathbf{C}^T$, $\mathbf{B}^T$, $\mathbf{A}^T$ be correct?

*Your answer:*

### C2 · Norms and the unit ball *(20 min)*

*(Week 2 exercise 6.)* For $\mathbf{z} = [3, -4, 12]^T$ compute the $\ell_1$, $\ell_2$, and $\ell_\infty$ norms **by hand** and enter the numbers. Then compute the Frobenius norm of the given matrix in code.

In [ ]:
z_vec = np.array([3.0, -4.0, 12.0])
l1_by_hand   = None
l2_by_hand   = None
linf_by_hand = None

M = np.array([[1.0, 2.0], [3.0, -1.0], [0.0, 2.0]])
frobenius = None                              # YOUR CODE HERE, without np.linalg.norm

The Week 2 notebook draws the unit ball $\{\mathbf{z} : \|\mathbf{z}\|_p = 1\}$ for $p = 1, 2, \infty$. Extend `unit_ball` so it also handles $p = 0.5$ and plot it alongside $p = 1$ and $p = 2$.

In [ ]:
theta = np.linspace(0, 2 * np.pi, 721)
dirs = np.stack([np.cos(theta), np.sin(theta)])        # unit directions, shape (2, 721)

def unit_ball(p):
    """Rescale each direction so that its p-'norm' is exactly one. Must work for any p > 0, including p = 0.5."""
    scale = None                                        # YOUR CODE HERE
    return dirs / scale

fig, ax = plt.subplots(figsize=(4.6, 4.6))
for p, color in [(0.5, "#7C3AED"), (1, ACCENT), (2, NDSU_GREEN)]:
    try:
        ball = unit_ball(p)
        ax.plot(ball[0], ball[1], color=color, lw=2, label=f"$p = {p}$")
    except Exception as e:
        print(f"p = {p}: {type(e).__name__}: {e}")
ax.set_aspect("equal"); ax.set_xlim(-1.45, 1.45); ax.set_ylim(-1.45, 1.45)
ax.set_xlabel("$z_1$"); ax.set_ylabel("$z_2$"); ax.legend(loc="upper right"); plt.show()

In [ ]:
check_close("C2 l1 by hand", l1_by_hand, np.sum(np.abs(z_vec)), tol=1e-9, hint="Sum of absolute values.")
check_close("C2 l2 by hand", l2_by_hand, np.sqrt(np.sum(z_vec ** 2)), tol=1e-9, hint="Square root of the sum of squares.")
check_close("C2 l_inf by hand", linf_by_hand, np.max(np.abs(z_vec)), tol=1e-9, hint="Largest absolute entry.")
check_close("C2 Frobenius", frobenius, np.linalg.norm(M), tol=1e-9, hint="Euclidean norm of the flattened matrix.")
try:
    _b = unit_ball(0.5)
    _ok = np.allclose(np.sum(np.abs(_b) ** 0.5, axis=0), 1.0) and np.all(np.sum(np.abs(_b), axis=0) <= 1 + 1e-9)
except Exception:
    _ok = False
check("C2 p = 0.5 ball", _ok, "scale = (sum |d|^p)^(1/p) along axis 0 works for every p > 0.")

**Written.** The $p = 0.5$ curve is not convex. Which property of a norm fails for $p < 1$, and why would that matter if you used $\|\cdot\|_{0.5}$ as a regularization penalty in Chapter 9? Two or three sentences.

*Your answer:*

### C3 · Dot products measure alignment *(15 min)*

*(Week 2 exercise 3.)* Two vibration spectra $\mathbf{a}$ and $\mathbf{b}$ have $\|\mathbf{a}\| = 3$, $\|\mathbf{b}\| = 5$, and $\mathbf{a}^T\mathbf{b} = 7.5$. A third spectrum is $\mathbf{c} = 4\mathbf{a}$.

Using $\mathbf{a}^T\mathbf{b} = \|\mathbf{a}\|\,\|\mathbf{b}\|\cos\theta$ (B.12), fill in the numbers by hand, in degrees. Then implement `angle_between` and confirm on concrete vectors that satisfy the given lengths.

In [ ]:
angle_ab_deg = None          # angle between a and b
a_dot_c      = None          # a^T c
angle_ac_deg = None          # angle between a and c

# Which quantity would you use to compare spectra recorded at different intensities?  "dot product" or "angle"
compare_with = None

def angle_between(u, v):
    """Angle between two vectors, in degrees."""
    return None              # YOUR CODE HERE

a = np.array([3.0, 0.0])
b = np.array([2.5, np.sqrt(25 - 2.5 ** 2)])     # ||b|| = 5 and a.b = 7.5
c = 4 * a
print("angle(a, b) =", angle_between(a, b), " angle(a, c) =", angle_between(a, c))

In [ ]:
check_close("C3 angle(a, b)", angle_ab_deg, 60.0, tol=1e-2, hint="cos(theta) = 7.5 / (3 * 5).")
check_close("C3 a.c", a_dot_c, 36.0, tol=1e-6, hint="a^T(4a) = 4 ||a||^2.")
check_close("C3 angle(a, c)", angle_ac_deg, 0.0, tol=1e-2, hint="c points the same way as a.")
check_answer("C3 what to compare with", compare_with, "49387897bee2",
             "Which of the two numbers stayed the same when c = 4a scaled the intensity?")
check_close("C3 angle_between", None if angle_between(a, b) is None else [angle_between(a, b), angle_between(a, c)],
            [60.0, 0.0], tol=1e-6, hint="arccos of the dot product divided by both norms, converted to degrees.")

### C4 · Column space and nullspace *(15 min)*

*(Week 2 exercise 4.)* A matrix $\mathbf{A} \in \mathbb{R}^{3\times5}$ maps a 5-dimensional input to a 3-dimensional output. State the **largest possible dimension of its column space** and the **smallest possible dimension of its nullspace**. Then, for a random such matrix, compute the rank and find a basis of the nullspace with `scipy.linalg.null_space`, and verify that $\mathbf{A}\mathbf{n} = \mathbf{0}$ for every basis vector.

In [ ]:
from scipy.linalg import null_space

max_column_space_dim = None
min_nullspace_dim    = None

A35 = rng.standard_normal((3, 5))
rank = None                            # YOUR CODE HERE  (np.linalg.matrix_rank)
N = None                               # YOUR CODE HERE  (null_space(A35), shape (5, nullity))

if N is not None:
    print("rank", rank, " nullspace basis shape", N.shape)
    print("A @ N, should be all zeros:\n", np.round(A35 @ N, 10))

In [ ]:
check_close("C4 max column space dimension", max_column_space_dim, 3, tol=0, hint="Column space lives in the output space R^3.")
check_close("C4 min nullspace dimension", min_nullspace_dim, 2, tol=0, hint="rank + nullity = number of columns.")
check_close("C4 rank", rank, np.linalg.matrix_rank(A35), tol=0, hint="np.linalg.matrix_rank.")
check("C4 nullspace basis", None if N is None else N.shape == (5, 2) and np.allclose(A35 @ N, 0),
      "null_space(A35) returns an orthonormal basis of {x : A x = 0}.")

**Written.** Give one concrete consequence for a sensor-fusion model that compresses 5 measurements into 3 features with a linear layer. What kind of change in the measurements can it never detect?

*Your answer:*

### C5 · Eigenspectrum, determinant, trace, and structured matrices *(25 min)*

*(Week 2 exercise 5.)* A $2\times2$ matrix has $\det\mathbf{A} = 0.01$ and $\operatorname{tr}\mathbf{A} = 2.1$. Since the determinant is the product of the eigenvalues and the trace is their sum, the eigenvalues are the roots of $\lambda^2 - 2.1\lambda + 0.01 = 0$. Compute them by hand (or with `np.roots`), then build a symmetric matrix with exactly those eigenvalues and watch what it does to the unit circle.

In [ ]:
eigenvalues = None                        # YOUR CODE HERE: a list or array of the two eigenvalues

# One of: "uniform shrink", "uniform stretch", "stretch one direction, collapse the other", "pure rotation"
what_it_does = None

if eigenvalues is not None:
    lam = np.sort(np.asarray(eigenvalues, dtype=float))[::-1]
    t = np.deg2rad(30)
    R = np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])
    A2 = R @ np.diag(lam) @ R.T                     # symmetric, eigenvalues exactly lam
    circle = np.stack([np.cos(theta), np.sin(theta)])
    ellipse = A2 @ circle
    fig, ax = plt.subplots(figsize=(4.6, 4.6))
    ax.plot(circle[0], circle[1], color="0.6", label="unit circle")
    ax.plot(ellipse[0], ellipse[1], color=NDSU_GREEN, lw=2, label="A · circle")
    ax.set_aspect("equal"); ax.legend(); plt.show()
    print("det", np.linalg.det(A2).round(4), " trace", np.trace(A2).round(4),
          " singular values", np.linalg.svd(A2, compute_uv=False).round(4))

Now the structured matrices of §B.4. Build a random **orthogonal** matrix with `np.linalg.qr`, and write the **permutation** matrix of Prince's (B.16) that sends $[a, b, c]^T$ to $[b, c, a]^T$. The check confirms the shortcut each one buys: $\mathbf{Q}^{-1} = \mathbf{Q}^T$, every $|\lambda_i| = 1$, $|\det| = 1$.

In [ ]:
Q, _ = np.linalg.qr(rng.standard_normal((5, 5)))    # Q is orthogonal

Q_inverse = None                                     # YOUR CODE HERE, without np.linalg.inv

P = None                                             # YOUR CODE HERE: 3 x 3 permutation matrix, P @ [a, b, c] = [b, c, a]

if Q_inverse is not None:
    print("Q^-1 Q = I :", np.allclose(Q_inverse @ Q, np.eye(5)))
    print("|eigenvalues of Q| :", np.abs(np.linalg.eigvals(Q)).round(6), " det:", np.linalg.det(Q).round(6))
if P is not None:
    print("P @ [a, b, c] ->", (P @ np.array([1, 2, 3])).astype(int), "(1 = a, 2 = b, 3 = c)")

In [ ]:
_lam = np.sort(np.roots([1, -2.1, 0.01]))[::-1]
check_close("C5 eigenvalues", None if eigenvalues is None else np.sort(np.asarray(eigenvalues, float))[::-1], _lam, tol=1e-3,
            hint="Roots of lambda^2 - (trace) lambda + det = 0.")
check_answer("C5 what the map does", what_it_does, "420d9dfee725",
             "One eigenvalue is about 2.1, the other about 0.005. What happens to the two axes of the circle?")
check("C5 orthogonal inverse", None if Q_inverse is None else np.allclose(Q_inverse, np.linalg.inv(Q)),
      "For an orthogonal matrix the inverse is the transpose.")
check("C5 permutation matrix", None if P is None else np.array_equal(P @ np.array([1, 2, 3]), [2, 3, 1])
      and np.allclose(P @ P.T, np.eye(3)),
      "One 1 per row and column; row i picks which input lands in output i.")

### C6 · One layer, three ways *(15 min)*

The single most important object in the course is $\mathbf{y} = \boldsymbol{\phi}_0 + \boldsymbol{\Phi}\mathbf{z}$, i.e. $y_i = \phi_{i0} + \sum_j \phi_{ij} z_j$. Compute it (i) as the explicit sums of (B.18), (ii) in matrix form, and (iii) with an `nn.Linear` layer whose weights you copy in. All three must agree.

In [ ]:
Phi0 = np.array([0.5, -1.0, 2.0])                       # phi_10, phi_20, phi_30
Phi  = np.array([[ 1.0,  2.0, -1.0],
                 [ 0.0,  1.5,  0.5],
                 [-2.0,  0.0,  1.0]])
zin  = np.array([1.0, -2.0, 0.5])

# (i) scalar sums, one output at a time
y_scalar = np.zeros(3)
for i in range(3):
    y_scalar[i] = None                                  # YOUR CODE HERE: phi_i0 + sum_j phi_ij z_j

# (ii) matrix form
y_matrix = None                                         # YOUR CODE HERE

# (iii) nn.Linear with the same numbers
layer = nn.Linear(3, 3)
with torch.no_grad():
    pass                                                # YOUR CODE HERE: copy Phi into layer.weight and Phi0 into layer.bias
y_torch = layer(torch.tensor(zin, dtype=torch.float32)).detach().numpy()

print("scalar :", y_scalar); print("matrix :", y_matrix); print("torch  :", y_torch)

In [ ]:
_y = Phi0 + Phi @ zin
check_close("C6 scalar sums", y_scalar, _y, tol=1e-6, hint="y_i = phi_i0 + sum over j of phi_ij * z_j.")
check_close("C6 matrix form", y_matrix, _y, tol=1e-6, hint="Phi0 + Phi @ z.")
check_close("C6 nn.Linear", y_torch, _y, tol=1e-5, hint="layer.weight is Phi (outputs x inputs) and layer.bias is phi_0; copy them inside torch.no_grad().")

### C7 · Matrix calculus and autograd, minding the layout *(20 min)*

*(Week 2 §6; exercise 8.)* Prince's table gives $\partial\mathbf{y}/\partial\mathbf{x}$ for $\mathbf{y} = \mathbf{A}\mathbf{x}$ as $\mathbf{A}^T$, with element $(i,j)$ equal to $\partial y_j/\partial x_i$ (inputs index the rows). PyTorch's `jacobian` uses the opposite layout and returns $\mathbf{A}$. Neither is wrong; they are transposes.

1. Compute the PyTorch Jacobian of $\mathbf{y} = \mathbf{A}\mathbf{x}$ and produce Prince's $\partial\mathbf{y}/\partial\mathbf{x}$ from it.
2. Use autograd to get $\partial \|\mathbf{x}\|^2/\partial\mathbf{x}$ and compare with $2\mathbf{x}$.
3. Use autograd on the least-squares loss of B2 at `phi_test` and compare with your `grad`.

In [ ]:
from torch.autograd.functional import jacobian

A_t = torch.tensor([[2.0, -1.0, 0.5],
                    [0.0,  3.0, 1.0]])                  # 2 x 3, so y in R^2 and x in R^3
x_t = torch.tensor([1.0, -2.0, 3.0])

J_torch = None                                          # YOUR CODE HERE: jacobian(lambda x: A_t @ x, x_t)
prince_dy_dx = None                                     # YOUR CODE HERE: Prince's layout, shape (3, 2)

x_req = torch.tensor([1.0, -2.0, 3.0], requires_grad=True)
grad_sq_norm = None                                     # YOUR CODE HERE: backward through ||x||^2, then x_req.grad

phi_req = torch.tensor(phi_test, requires_grad=True)
X_t, y_t = torch.tensor(X), torch.tensor(y)
grad_loss_autograd = None                               # YOUR CODE HERE: backward through sum((X phi - y)^2), then phi_req.grad

print("torch jacobian:\n", J_torch); print("Prince's dy/dx:\n", prince_dy_dx)
print("d||x||^2/dx:", grad_sq_norm, " expected:", 2 * x_t)
print("autograd loss gradient:", grad_loss_autograd, "\nyour grad from B2     :", grad(phi_test, X, y))

In [ ]:
check_close("C7 torch jacobian equals A", None if J_torch is None else J_torch.numpy(), A_t.numpy(), tol=1e-6,
            hint="jacobian(f, x) where f(x) = A_t @ x.")
check_close("C7 Prince's layout equals A^T", None if prince_dy_dx is None else np.asarray(prince_dy_dx), A_t.numpy().T, tol=1e-6,
            hint="Inputs index the rows in Prince's table: transpose the PyTorch answer.")
check_close("C7 gradient of ||x||^2", None if grad_sq_norm is None else np.asarray(grad_sq_norm), 2 * x_t.numpy(), tol=1e-6,
            hint="torch.sum(x_req ** 2).backward(); the gradient lands in x_req.grad.")
check_close("C7 autograd matches hand gradient", None if grad_loss_autograd is None else np.asarray(grad_loss_autograd),
            grad(phi_test, X, y), tol=1e-5, hint="Build the loss from X_t, phi_req, y_t and call .backward().")

### C8 · Why gradient descent stalls: conditioning *(25 min)*

*(Week 2 §8.5; exercise 7.)* The least-squares loss is a quadratic bowl whose curvature is the Hessian $\partial^2 L/\partial\boldsymbol{\phi}^2 = 2\mathbf{X}^T\mathbf{X}$. Its largest eigenvalue sets the largest stable step size, $2/\lambda_{\max}$; its smallest sets how fast the valley floor is crossed; their ratio is the **condition number**.

Compare three encodings of the same input: raw degrees Celsius, **kelvin** (add 273.15), and standardized. For each, compute the Hessian eigenvalues, the condition number, and the number of gradient descent steps needed to get within 1% of the optimal loss when the step size is 90% of the stable limit. The descent routine is given.

In [ ]:
def gradient_descent(M, target, lr, n_steps):
    """Plain gradient descent on ||M phi - target||^2 from the origin; returns (phi, loss history)."""
    phi = np.zeros(M.shape[1]); hist = []
    for _ in range(n_steps):
        phi = phi - lr * (2 * M.T @ (M @ phi - target))
        hist.append(np.sum((M @ phi - target) ** 2))
    return phi, np.array(hist)

def steps_to_1pct(hist, best):
    ok = hist <= 1.01 * best
    return int(np.argmax(ok)) + 1 if ok.any() else None

encodings = {
    "celsius":      np.column_stack([np.ones(I), temp]),
    "kelvin":       None,                              # YOUR CODE HERE
    "standardized": Z,
}

report = {}
for name, M in encodings.items():
    if M is None:
        continue
    ev = None                                          # YOUR CODE HERE: eigenvalues of the Hessian 2 M^T M  (np.linalg.eigvalsh)
    cond = None                                        # YOUR CODE HERE: largest / smallest eigenvalue
    lr_ = None                                         # YOUR CODE HERE: 0.9 * 2 / largest eigenvalue
    if cond is None or lr_ is None:
        print(f"{name}: fill in ev, cond, and lr_ first"); continue
    best = np.sum((M @ np.linalg.solve(M.T @ M, M.T @ y) - y) ** 2)
    _, hist = gradient_descent(M, y, lr_, 5_000)
    report[name] = (cond, lr_, steps_to_1pct(hist, best))

print(f"{'encoding':<14}{'condition number':>18}{'stable step':>14}{'steps to 1%':>14}")
for name, (cond, lr_, n) in report.items():
    print(f"{name:<14}{cond:>18,.1f}{lr_:>14.3e}{str(n):>14}")

In [ ]:
def _cond(M):
    ev = np.linalg.eigvalsh(2 * M.T @ M); return ev[-1] / ev[0]
_ok = all(k in report for k in ["celsius", "kelvin", "standardized"])
check("C8 all three encodings reported", None if not report else _ok, "Build the kelvin design matrix and fill ev, cond, lr_.")
if _ok:
    check_close("C8 celsius condition number", report["celsius"][0], _cond(encodings["celsius"]), tol=1e-3,
                hint="Largest eigenvalue over smallest of 2 M^T M.")
    check_close("C8 kelvin condition number", report["kelvin"][0], _cond(np.column_stack([np.ones(I), temp + 273.15])), tol=1e-3,
                hint="Add 273.15 to the temperature column only, not the column of ones.")
    check("C8 kelvin is worse than celsius", report["kelvin"][0] > report["celsius"][0],
          "A larger offset makes the two columns more nearly parallel.")
    check_close("C8 standardized condition number is 1", report["standardized"][0], 1.0, tol=1e-6,
                hint="With a centered, unit-variance column the two columns are orthogonal and equal in length.")
    check("C8 standardized converges, kelvin does not",
          report["standardized"][2] is not None and report["standardized"][2] < 100 and report["kelvin"][2] is None,
          "Compare the steps-to-1% column.")

**Written.** The closed-form solution is the same line in every encoding, but gradient descent goes from fine to crippled. Explain why, using the words *eigenvalues*, *stable step size*, and *valley floor*. Then name the cheapest fix and the chapter that is entirely devoted to the expensive ones.

*Your answer:*

## Part D — Self-assessment

### D1 · Where you stand *(10 min)*

Run the summary, then rate yourself honestly on the learning objectives from the two notes. This is the part of the assignment that most directly shapes what we spend class time on in Week 3.

In [ ]:
summary()

For each objective, replace `?` with **3** (could teach it), **2** (can do it with the notes open), or **1** (not yet).

| # | Objective | Rating |
|:--|:--|:--:|
| 1 | Distinguish supervised, unsupervised, and reinforcement learning, and classify a new problem | ? |
| 2 | Tell regression, multivariate regression, binary and multiclass classification apart from the output | ? |
| 3 | Explain what makes an input hard and why that shapes model design | ? |
| 4 | Say precisely what a model, parameters, loss, training, inference, and test data are | ? |
| 5 | Explain what latent variables buy a generative model | ? |
| 6 | State the temporal credit assignment problem and the exploration–exploitation trade-off | ? |
| 7 | Compute transposes, norms, products, dot products, and inverses, and say what each costs | ? |
| 8 | Explain what the eigenspectrum, determinant, and trace say about a linear map | ? |
| 9 | Write a system of linear equations as $\mathbf{y} = \boldsymbol{\phi}_0 + \boldsymbol{\Phi}\mathbf{z}$ and find it inside `nn.Linear` | ? |
| 10 | State the shape of $\partial\mathbf{y}/\partial\mathbf{x}$ and obtain it with autograd | ? |
| 11 | Derive the least-squares gradient and train linear regression three ways | ? |
| 12 | Explain, via the eigenspectrum of the loss surface, why rescaling an input changes convergence | ? |

**Muddiest point.** Which single idea from Weeks 1–2 is still least clear to you? One or two sentences; be specific.

*Your answer:*

**Time spent** (hours, roughly): ?

---

*IMSE 774 · Neural Networks · North Dakota State University · Fall 2026. Course notes are licensed CC BY-NC-SA 4.0; code is MIT. Problems marked UDL are from Prince, Understanding Deep Learning (MIT Press, 2024), Chapter 2.*